In [4]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder

train_transaction = pd.read_csv('ieee-fraud-detection/train_transaction.csv')
train_identity    = pd.read_csv('ieee-fraud-detection/train_identity.csv')
test_transaction  = pd.read_csv('ieee-fraud-detection/test_transaction.csv')
test_identity     = pd.read_csv('ieee-fraud-detection/test_identity.csv')

train = train_transaction.merge(train_identity, on='TransactionID', how='left')
test  = test_transaction.merge(test_identity,  on='TransactionID', how='left')
test.columns = test.columns.str.replace('-', '_')

split = train['TransactionDT'].quantile(0.80)
tr  = train[train['TransactionDT'] <= split].copy()
val = train[train['TransactionDT'] >  split].copy()
print(f"tr {tr.shape} | val {val.shape} | test {test.shape}")

tr (472432, 434) | val (118108, 434) | test (506691, 433)


### Feature definition

In [5]:
delete_cols = ['id_29','V191','V196','V28','V241','V107','V117','V119','V120',
               'V113','V305','V88','V89','V27','V41','V65','V325','V327','V68','id_16','id_27']

cat_fea = ['ProductCD','card1','card2','card3','card4','card5','card6',
           'addr1','addr2','P_emaildomain','R_emaildomain','DeviceType','DeviceInfo']

for i in range(1, 10):
    if 'M'+str(i) not in delete_cols:
        cat_fea.append('M'+str(i))
for i in range(12, 39):
    if 'id_'+str(i) not in delete_cols:
        cat_fea.append('id_'+str(i))

num_fea = [f for f in train.columns
           if f not in ['TransactionID','isFraud'] + cat_fea + delete_cols]

### TransactionDT & TransactionAmt features

In [6]:
for df in [tr, val, test]:
    df['hour']          = (df['TransactionDT'] // 3600) % 24
    df['dayofweek']     = (df['TransactionDT'] // 86400) % 7
    df['day']           = df['TransactionDT']  // 86400
    df['hour_sin']      = np.sin(2 * np.pi * df['hour']      / 24)
    df['hour_cos']      = np.cos(2 * np.pi * df['hour']      / 24)
    df['dayofweek_sin'] = np.sin(2 * np.pi * df['dayofweek'] / 7)
    df['dayofweek_cos'] = np.cos(2 * np.pi * df['dayofweek'] / 7)

    df['log_amt']        = np.log1p(df['TransactionAmt'])
    df['amt_decimal']    = (df['TransactionAmt'] % 1).round(2).astype('category')
    df['amt_is_round']   = (df['TransactionAmt'] % 1 == 0).astype(np.int8)
    df['amt_last_digit'] = pd.Series(np.where(
        df['TransactionAmt'] % 1 == 0,
        df['TransactionAmt'].astype(int) % 10, -1
    ), index=df.index).astype('category')

num_fea.extend(['hour','dayofweek','day','hour_sin','hour_cos',
                'dayofweek_sin','dayofweek_cos','log_amt','amt_is_round'])
cat_fea.extend(['amt_decimal','amt_last_digit'])

In [121]:
tr['amt_decimal'].nunique() 

101

### Log transform high-skew numeric features (|skew| > 10)

In [7]:
skew_series = tr[num_fea].select_dtypes(include=[np.number]).skew()
high_skew_cols = skew_series[skew_series.abs() > 10].index.tolist()
print(f"Log transforming {len(high_skew_cols)} features")

for col in high_skew_cols:
    for df in [tr, val, test]:
        df[col + '_log'] = np.log1p(df[col].clip(lower=0))
    num_fea.append(col + '_log')

Log transforming 195 features


### Bin D15

In [8]:
_, bin_edges = pd.qcut(tr['D15'], q=10, retbins=True, duplicates='drop')
for df in [tr, val, test]:
    df['D15_bin'] = pd.cut(df['D15'], bins=bin_edges, labels=False,
                           include_lowest=True).astype('category')
cat_fea.append('D15_bin')
print(f"D15 binned. Total so far — numeric: {len(num_fea)} | categorical: {len(cat_fea)}")

D15 binned. Total so far — numeric: 569 | categorical: 49


In [9]:
cat_feat_back = cat_fea
num_fea_back = num_fea
str( cat_fea)

"['ProductCD', 'card1', 'card2', 'card3', 'card4', 'card5', 'card6', 'addr1', 'addr2', 'P_emaildomain', 'R_emaildomain', 'DeviceType', 'DeviceInfo', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_13', 'id_14', 'id_15', 'id_17', 'id_18', 'id_19', 'id_20', 'id_21', 'id_22', 'id_23', 'id_24', 'id_25', 'id_26', 'id_28', 'id_30', 'id_31', 'id_32', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'amt_decimal', 'amt_last_digit', 'D15_bin']"

In [10]:
str(num_fea)

"['TransactionDT', 'TransactionAmt', 'dist1', 'dist2', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14', 'D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'D9', 'D10', 'D11', 'D12', 'D13', 'D14', 'D15', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V29', 'V30', 'V31', 'V32', 'V33', 'V34', 'V35', 'V36', 'V37', 'V38', 'V39', 'V40', 'V42', 'V43', 'V44', 'V45', 'V46', 'V47', 'V48', 'V49', 'V50', 'V51', 'V52', 'V53', 'V54', 'V55', 'V56', 'V57', 'V58', 'V59', 'V60', 'V61', 'V62', 'V63', 'V64', 'V66', 'V67', 'V69', 'V70', 'V71', 'V72', 'V73', 'V74', 'V75', 'V76', 'V77', 'V78', 'V79', 'V80', 'V81', 'V82', 'V83', 'V84', 'V85', 'V86', 'V87', 'V90', 'V91', 'V92', 'V93', 'V94', 'V95', 'V96', 'V97', 'V98', 'V99', 'V100', 'V101', 'V102', 'V103', 'V104', 'V105', 'V106', 'V108', 'V109', 'V110', 'V111', 'V112', 'V114', 'V115', 'V116', 'V118

In [68]:
tr["card1"]


0         13926
1          2755
2          4663
3         18132
4          4497
          ...  
472427    15484
472428    17188
472429    16659
472430     7919
472431    13749
Name: card1, Length: 472432, dtype: int64

### Sort by TransactionDT — required for all rolling features

In [118]:
tr_s   = tr.sort_values('TransactionDT').reset_index(drop=True)
val_s  = val.sort_values('TransactionDT').reset_index(drop=True)
test_s = test.sort_values('TransactionDT').reset_index(drop=True)

tr_s["cust_id"] = tr_s["card1"].astype("string") + "_" + tr_s["addr1"].astype("string") + "_" + tr_s["D1"].astype("string")
val_s["cust_id"] = val_s["card1"].astype("string") + "_" + val_s["addr1"].astype("string") + "_" + val_s["D1"].astype("string")
test_s["cust_id"] = test_s["card1"].astype("string") + "_" + test_s["addr1"].astype("string") + "_" + test_s["D1"].astype("string")

cat_fea = ['ProductCD', 'card1', 'card2', 'card3', 'card4', 'card5', 'card6', 'addr1', 'addr2', 'P_emaildomain', 'R_emaildomain', 'DeviceType', 'DeviceInfo', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_13', 'id_14', 'id_15', 'id_17', 'id_18', 'id_19', 'id_20', 'id_21', 'id_22', 'id_23', 'id_24', 'id_25', 'id_26', 'id_28', 'id_30', 'id_31', 'id_32', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'amt_decimal', 'amt_last_digit', 'D15_bin']
num_fea = ['TransactionDT', 'TransactionAmt', 'dist1', 'dist2', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14', 'D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'D9', 'D10', 'D11', 'D12', 'D13', 'D14', 'D15', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V29', 'V30', 'V31', 'V32', 'V33', 'V34', 'V35', 'V36', 'V37', 'V38', 'V39', 'V40', 'V42', 'V43', 'V44', 'V45', 'V46', 'V47', 'V48', 'V49', 'V50', 'V51', 'V52', 'V53', 'V54', 'V55', 'V56', 'V57', 'V58', 'V59', 'V60', 'V61', 'V62', 'V63', 'V64', 'V66', 'V67', 'V69', 'V70', 'V71', 'V72', 'V73', 'V74', 'V75', 'V76', 'V77', 'V78', 'V79', 'V80', 'V81', 'V82', 'V83', 'V84', 'V85', 'V86', 'V87', 'V90', 'V91', 'V92', 'V93', 'V94', 'V95', 'V96', 'V97', 'V98', 'V99', 'V100', 'V101', 'V102', 'V103', 'V104', 'V105', 'V106', 'V108', 'V109', 'V110', 'V111', 'V112', 'V114', 'V115', 'V116', 'V118', 'V121', 'V122', 'V123', 'V124', 'V125', 'V126', 'V127', 'V128', 'V129', 'V130', 'V131', 'V132', 'V133', 'V134', 'V135', 'V136', 'V137', 'V138', 'V139', 'V140', 'V141', 'V142', 'V143', 'V144', 'V145', 'V146', 'V147', 'V148', 'V149', 'V150', 'V151', 'V152', 'V153', 'V154', 'V155', 'V156', 'V157', 'V158', 'V159', 'V160', 'V161', 'V162', 'V163', 'V164', 'V165', 'V166', 'V167', 'V168', 'V169', 'V170', 'V171', 'V172', 'V173', 'V174', 'V175', 'V176', 'V177', 'V178', 'V179', 'V180', 'V181', 'V182', 'V183', 'V184', 'V185', 'V186', 'V187', 'V188', 'V189', 'V190', 'V192', 'V193', 'V194', 'V195', 'V197', 'V198', 'V199', 'V200', 'V201', 'V202', 'V203', 'V204', 'V205', 'V206', 'V207', 'V208', 'V209', 'V210', 'V211', 'V212', 'V213', 'V214', 'V215', 'V216', 'V217', 'V218', 'V219', 'V220', 'V221', 'V222', 'V223', 'V224', 'V225', 'V226', 'V227', 'V228', 'V229', 'V230', 'V231', 'V232', 'V233', 'V234', 'V235', 'V236', 'V237', 'V238', 'V239', 'V240', 'V242', 'V243', 'V244', 'V245', 'V246', 'V247', 'V248', 'V249', 'V250', 'V251', 'V252', 'V253', 'V254', 'V255', 'V256', 'V257', 'V258', 'V259', 'V260', 'V261', 'V262', 'V263', 'V264', 'V265', 'V266', 'V267', 'V268', 'V269', 'V270', 'V271', 'V272', 'V273', 'V274', 'V275', 'V276', 'V277', 'V278', 'V279', 'V280', 'V281', 'V282', 'V283', 'V284', 'V285', 'V286', 'V287', 'V288', 'V289', 'V290', 'V291', 'V292', 'V293', 'V294', 'V295', 'V296', 'V297', 'V298', 'V299', 'V300', 'V301', 'V302', 'V303', 'V304', 'V306', 'V307', 'V308', 'V309', 'V310', 'V311', 'V312', 'V313', 'V314', 'V315', 'V316', 'V317', 'V318', 'V319', 'V320', 'V321', 'V322', 'V323', 'V324', 'V326', 'V328', 'V329', 'V330', 'V331', 'V332', 'V333', 'V334', 'V335', 'V336', 'V337', 'V338', 'V339', 'id_01', 'id_02', 'id_03', 'id_04', 'id_05', 'id_06', 'id_07', 'id_08', 'id_09', 'id_10', 'id_11', 'hour', 'dayofweek', 'day', 'hour_sin', 'hour_cos', 'dayofweek_sin', 'dayofweek_cos', 'log_amt', 'amt_is_round', 'TransactionAmt_log', 'C1_log', 'C2_log', 'C3_log', 'C4_log', 'C6_log', 'C7_log', 'C8_log', 'C10_log', 'C11_log', 'C12_log', 'C14_log', 'V1_log', 'V14_log', 'V23_log', 'V37_log', 'V38_log', 'V44_log', 'V45_log', 'V47_log', 'V55_log', 'V56_log', 'V77_log', 'V78_log', 'V86_log', 'V87_log', 'V96_log', 'V99_log', 'V100_log', 'V101_log', 'V102_log', 'V103_log', 'V104_log', 'V105_log', 'V106_log', 'V108_log', 'V109_log', 'V110_log', 'V111_log', 'V112_log', 'V114_log', 'V116_log', 'V118_log', 'V121_log', 'V122_log', 'V123_log', 'V125_log', 'V126_log', 'V127_log', 'V128_log', 'V129_log', 'V130_log', 'V131_log', 'V132_log', 'V133_log', 'V134_log', 'V135_log', 'V136_log', 'V137_log', 'V138_log', 'V142_log', 'V161_log', 'V162_log', 'V163_log', 'V166_log', 'V167_log', 'V168_log', 'V172_log', 'V176_log', 'V177_log', 'V178_log', 'V179_log', 'V180_log', 'V182_log', 'V183_log', 'V186_log', 'V187_log', 'V190_log', 'V192_log', 'V193_log', 'V198_log', 'V199_log', 'V200_log', 'V201_log', 'V202_log', 'V203_log', 'V204_log', 'V205_log', 'V206_log', 'V207_log', 'V208_log', 'V209_log', 'V210_log', 'V211_log', 'V212_log', 'V213_log', 'V214_log', 'V215_log', 'V216_log', 'V218_log', 'V219_log', 'V220_log', 'V221_log', 'V222_log', 'V223_log', 'V224_log', 'V225_log', 'V226_log', 'V227_log', 'V228_log', 'V229_log', 'V230_log', 'V231_log', 'V232_log', 'V236_log', 'V240_log', 'V242_log', 'V243_log', 'V244_log', 'V245_log', 'V246_log', 'V247_log', 'V248_log', 'V249_log', 'V252_log', 'V253_log', 'V254_log', 'V255_log', 'V256_log', 'V257_log', 'V258_log', 'V259_log', 'V261_log', 'V262_log', 'V263_log', 'V264_log', 'V265_log', 'V266_log', 'V267_log', 'V268_log', 'V269_log', 'V270_log', 'V271_log', 'V272_log', 'V273_log', 'V274_log', 'V275_log', 'V276_log', 'V277_log', 'V278_log', 'V279_log', 'V280_log', 'V283_log', 'V285_log', 'V287_log', 'V290_log', 'V291_log', 'V292_log', 'V293_log', 'V294_log', 'V295_log', 'V296_log', 'V297_log', 'V298_log', 'V299_log', 'V300_log', 'V301_log', 'V306_log', 'V307_log', 'V308_log', 'V309_log', 'V310_log', 'V311_log', 'V312_log', 'V313_log', 'V314_log', 'V315_log', 'V316_log', 'V317_log', 'V318_log', 'V319_log', 'V320_log', 'V321_log', 'V329_log', 'V331_log', 'V332_log', 'V333_log', 'V334_log', 'V335_log', 'V336_log', 'V337_log', 'V338_log', 'V339_log', 'id_04_log', 'id_10_log']
val_database = pd.concat([tr_s, val_s], axis=0).sort_values('TransactionDT').reset_index(drop=True)
test_database = pd.concat([tr_s, val_s,test_s], axis=0).sort_values('TransactionDT').reset_index(drop=True)

### 24h rolling fraud features per entity

In [119]:
def rolling_cnt_value(base_df, target_df, col, window, exclude_self=False, value_col=None):
    """
    For each row in target_df, aggregate rows in base_df with the same col value
    within [TransactionDT - window, TransactionDT).

    value_col=None  → returns counts only (int32 array)
    value_col='x'   → returns (counts, sums, avg, min, max, p25, p50, p75)
                       all stats computed over base_df['x'] in the window
                       nan when count == 0

    exclude_self=True uses (dt - 1) as upper bound to exclude the current row.
    base_df must be sorted by TransactionDT.
    """
    needed   = [col, 'TransactionDT'] + ([value_col] if value_col else [])
    base_s   = base_df[needed].sort_values('TransactionDT').reset_index(drop=True)
    target_s = target_df[[col, 'TransactionDT']].reset_index(drop=True).copy()
    target_s['_pos'] = np.arange(len(target_s))

    n_target = len(target_s)
    counts = np.zeros(n_target, dtype=np.int32)

    if value_col:
        sums = np.full(n_target, np.nan, dtype=np.float32)
        avgs = np.full(n_target, np.nan, dtype=np.float32)
        mins = np.full(n_target, np.nan, dtype=np.float32)
        maxs = np.full(n_target, np.nan, dtype=np.float32)
        p25s = np.full(n_target, np.nan, dtype=np.float32)
        p50s = np.full(n_target, np.nan, dtype=np.float32)
        p75s = np.full(n_target, np.nan, dtype=np.float32)

    for entity, grp in target_s.groupby(col):
        base_grp = base_s[base_s[col] == entity]
        dt_base  = base_grp['TransactionDT'].values
        if len(dt_base) == 0:
            continue
        dt_target = grp['TransactionDT'].values
        upper = dt_target - 1 if exclude_self else dt_target
        right = np.searchsorted(dt_base, upper,              side='right')
        left  = np.searchsorted(dt_base, dt_target - window, side='left')
        pos   = grp['_pos'].values
        counts[pos] = right - left

        if value_col:
            vals = base_grp[value_col].values
            for i, (l, r, p) in enumerate(zip(left, right, pos)):
                if r > l:
                    window_vals = vals[l:r].astype(np.float32)
                    sums[p] = window_vals.sum()
                    avgs[p] = window_vals.mean()
                    mins[p] = window_vals.min()
                    maxs[p] = window_vals.max()
                    p25s[p] = np.percentile(window_vals, 25)
                    p50s[p] = np.percentile(window_vals, 50)
                    p75s[p] = np.percentile(window_vals, 75)

    if value_col:
        return counts, sums, avgs, mins, maxs,p25s,p50s,p75s
    return counts


# ── set your desired time windows here (in seconds) ──────────────────────────
windows = [86400, 86400*7, 86400*14, 86400*30,86400*60, 86400*90, 86400*120, 86400*180]   # 1d / 7d / 14d / 30d
n = 128
# ─────────────────────────────────────────────────────────────────────────────

### Categorical encoding\n\n- Unique values < 128 → native LightGBM category dtype\n- Unique values ≥ 128 → `rolling_cnt_value` captures windowed frequency; original column dropped from features

In [120]:
rolling_count_features = []

for col in list(cat_fea):
    cur_unique = tr_s[col].nunique()
    if col in ["card1","card2","addr1","P_emaildomain","card5"]:
        for w in windows:
            suffix = f"{col}_{w}s"
            feats  = [f"{suffix}_cnt", f"{suffix}_sum", f"{suffix}_avg",
                      f"{suffix}_min", f"{suffix}_max", f"{suffix}_p25", f"{suffix}_p50", f"{suffix}_p75"]
    
            for df, base_df, excl in [(tr_s,   tr_s,          True),
                                       (val_s,  val_database,  True),
                                       (test_s, test_database,  True)]:
                cnt, sums, avg, mn, mx, p25, p50, p75 = rolling_cnt_value(
                    base_df, df, col, window=w, exclude_self=excl, value_col='TransactionAmt'
                )
                df[feats[0]] = cnt
                df[feats[1]] = sums
                df[feats[2]] = avg
                df[feats[3]] = mn
                df[feats[4]] = mx
                df[feats[5]] = p25
                df[feats[6]] = p50
                df[feats[7]] = p75
    
            rolling_count_features.extend(feats)
            print(f"  {suffix} done")
    
        cat_fea.remove(col)
    else:
        if cur_unique < n:
            tr_cats = tr_s[col].astype('category').cat.categories
            for df in [tr_s, val_s, test_s]:
                df[col] = pd.Categorical(df[col], categories=tr_cats)
        else:
            for w in windows:
                for df, base_df, excl in [(tr_s,   tr_s,          True),
                                       (val_s,  val_database,  True),
                                       (test_s, test_database,  True)]:
                    cnt = rolling_cnt_value(base_df, df, col, window=w, exclude_self=excl)
                    df[col+"_cnt"] = cnt
    
            rolling_count_features.append(col+"_cnt")
            print(f"  {col} done")
    
        cat_fea.remove(col)
                


  card1_86400s done
  card1_604800s done
  card1_1209600s done
  card1_2592000s done
  card1_5184000s done
  card1_7776000s done
  card1_10368000s done
  card1_15552000s done
  card2_86400s done
  card2_604800s done
  card2_1209600s done
  card2_2592000s done
  card2_5184000s done
  card2_7776000s done
  card2_10368000s done
  card2_15552000s done
  card5_86400s done
  card5_604800s done
  card5_1209600s done
  card5_2592000s done
  card5_5184000s done
  card5_7776000s done
  card5_10368000s done
  card5_15552000s done
  addr1_86400s done
  addr1_604800s done
  addr1_1209600s done
  addr1_2592000s done
  addr1_5184000s done
  addr1_7776000s done
  addr1_10368000s done


KeyboardInterrupt: 

In [ ]:
cat_fea

In [107]:
features = num_fea + cat_fea + rolling_count_features
print(f"Total features: {len(features)} | numeric: {len(num_fea)} | categorical: {len(cat_fea)}")
print(f"Categorical features: {cat_fea}")

Total features: 969 | numeric: 569 | categorical: 40
Categorical features: ['ProductCD', 'card3', 'card4', 'card5', 'card6', 'addr2', 'P_emaildomain', 'R_emaildomain', 'DeviceType', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_13', 'id_14', 'id_15', 'id_17', 'id_18', 'id_22', 'id_23', 'id_24', 'id_26', 'id_28', 'id_30', 'id_31', 'id_32', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'amt_decimal', 'amt_last_digit', 'D15_bin']


### Train model

In [110]:
X_tr   = tr_s[features].copy()
X_val  = val_s[features].copy()
X_test = test_s[features].copy()
y_tr   = tr_s['isFraud']
y_val  = val_s['isFraud']

model = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=256,
    random_state=42,
    n_jobs=-1
)
model.fit(
    X_tr, y_tr,
    eval_set=[(X_tr, y_tr), (X_val, y_val)],
    eval_names=["train", "valid"],
    categorical_feature=cat_fea,
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)]
)

[LightGBM] [Info] Number of positive: 16599, number of negative: 455833
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.333420 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 134401
[LightGBM] [Info] Number of data points in the train set: 472432, number of used features: 969
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.035135 -> initscore=-3.312784
[LightGBM] [Info] Start training from score -3.312784
Training until validation scores don't improve for 50 rounds
[100]	train's binary_logloss: 0.0320115	valid's binary_logloss: 0.0894418
Early stopping, best iteration is:
[134]	train's binary_logloss: 0.0242311	valid's binary_logloss: 0.0888594


,boosting_type,'gbdt'
,num_leaves,256
,max_depth,-1
,learning_rate,0.05
,n_estimators,500
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


### Validation performance

In [111]:
val_pred = model.predict_proba(X_val)[:, 1]
auc = roc_auc_score(y_val, val_pred)
print(f"model_v2 AUC : 0.9257  (label_fre_3, global freq — leaked)")
print(f"model_v3 AUC : {auc:.4f} (rolling count freq — leak-free)")
print(f"Difference   : {auc - 0.9257:+.4f}")

model_v2 AUC : 0.9257  (label_fre_3, global freq — leaked)
model_v3 AUC : 0.9156 (rolling count freq — leak-free)
Difference   : -0.0101


### Feature importance

In [112]:
importance = (pd.DataFrame({
    'feature':    features,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False).reset_index(drop=True))

print("=== Top 20 ===")
print(importance.head(40).to_string(index=False))

new_feats = (['hour','dayofweek','day','hour_sin','hour_cos','dayofweek_sin','dayofweek_cos',
              'log_amt','amt_is_round','amt_decimal','amt_last_digit','D15_bin']
             + rolling_count_features )
print("\n=== New engineered features ===")
print(importance[importance['feature'].isin(new_feats)].to_string(index=False))

=== Top 20 ===
           feature  importance
       amt_decimal        1791
    TransactionAmt         692
               C13         501
     P_emaildomain         482
             id_31         464
               D15         422
                D2         372
                C1         365
             dist1         363
             card5         350
                D1         291
     TransactionDT         279
                D4         274
               D10         257
             id_30         254
                C6         243
               C14         236
                C2         233
addr1_2592000s_max         218
 addr1_604800s_avg         216
card1_2592000s_max         214
               C11         213
card2_2592000s_avg         206
addr1_1209600s_avg         202
addr1_1209600s_max         199
 card2_604800s_avg         196
card2_1209600s_max         196
 card2_604800s_sum         191
 addr1_604800s_max         191
  addr1_86400s_max         187
addr1_5184000s_avg      

### Generate submission

In [113]:
test_pred = model.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({
    'TransactionID': test_s['TransactionID'].values,
    'isFraud':       test_pred
})

test_original_order = test_transaction['TransactionID'].values
submission = submission.set_index('TransactionID').loc[test_original_order].reset_index()

submission.to_csv('submission_v4.csv', index=False)
print(f"Saved: {submission.shape}")
print(submission['isFraud'].describe().round(4))

Saved: (506691, 2)
count    506691.0000
mean          0.0289
std           0.1117
min           0.0005
25%           0.0028
50%           0.0051
75%           0.0116
max           0.9952
Name: isFraud, dtype: float64
